# Amazon ML Challenge - Approach 9: Informer with Text + Image Embeddings

This notebook implements an Informer (Information Transformer) based approach combining:
- Text embeddings from catalog_content
- Pre-computed EfficientNet-B0 image embeddings
- Log-transformed price prediction
- ProbSparse self-attention for efficient long-term dependencies

In [ ]:
# Import required libraries
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from sklearn.preprocessing import StandardScaler
import optuna
from optuna.visualization import plot_optimization_history, plot_param_importances
import math
import warnings
warnings.filterwarnings('ignore')

# Set random seeds for reproducibility
np.random.seed(42)
torch.manual_seed(42)
if torch.cuda.is_available():
    torch.cuda.manual_seed(42)

# Device configuration
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

## 1. Data Loading and Preprocessing

In [ ]:
# Configuration
BASE_DIR = '/Users/gabi/Desktop/DL stuff/AMAZONML/student_resource'
TRAIN_CSV = f'{BASE_DIR}/dataset/train.csv'
TEST_CSV = f'{BASE_DIR}/dataset/test.csv'

# Paths for pre-computed embeddings (adjust these to your actual paths)
TRAIN_IMAGE_EMBEDDINGS = 'train_image_embeddings.npy'  # Your pre-saved train image embeddings
TEST_IMAGE_EMBEDDINGS = 'test_image_embeddings.npy'    # Your pre-saved test image embeddings

# Load datasets
print("Loading datasets...")
train_df = pd.read_csv(TRAIN_CSV)
test_df = pd.read_csv(TEST_CSV)

print(f"Train shape: {train_df.shape}")
print(f"Test shape: {test_df.shape}")
print(f"\nTrain columns: {train_df.columns.tolist()}")
print(f"\nFirst few rows:")
print(train_df.head())

In [ ]:
# Preprocess catalog_content text
def preprocess_text(text):
    """Clean and preprocess catalog content"""
    if pd.isna(text):
        return ""
    text = str(text).lower()
    # Basic cleaning
    text = text.replace('\n', ' ').replace('\t', ' ')
    # Remove extra spaces
    text = ' '.join(text.split())
    return text

print("Preprocessing text...")
train_df['clean_text'] = train_df['catalog_content'].apply(preprocess_text)
test_df['clean_text'] = test_df['catalog_content'].apply(preprocess_text)

print(f"Sample preprocessed text:")
print(train_df['clean_text'].iloc[0][:200])

## 2. Create Text Embeddings using TF-IDF

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer

# Create TF-IDF text embeddings
print("Creating TF-IDF text embeddings...")
tfidf_vectorizer = TfidfVectorizer(
    max_features=512,  # Dimension of text embeddings
    ngram_range=(1, 2),
    min_df=2,
    max_df=0.95
)

# Fit on train and transform both train and test
train_text_embeddings = tfidf_vectorizer.fit_transform(train_df['clean_text']).toarray()
test_text_embeddings = tfidf_vectorizer.transform(test_df['clean_text']).toarray()

print(f"Train text embeddings shape: {train_text_embeddings.shape}")
print(f"Test text embeddings shape: {test_text_embeddings.shape}")

## 3. Load Pre-computed Image Embeddings

In [ ]:
# Load pre-computed image embeddings
print("Loading pre-computed image embeddings...")
train_image_embeddings = np.load(TRAIN_IMAGE_EMBEDDINGS)
test_image_embeddings = np.load(TEST_IMAGE_EMBEDDINGS)

print(f"Train image embeddings shape: {train_image_embeddings.shape}")
print(f"Test image embeddings shape: {test_image_embeddings.shape}")

# Verify dimensions match
assert train_image_embeddings.shape[0] == len(train_df), "Train embeddings count mismatch"
assert test_image_embeddings.shape[0] == len(test_df), "Test embeddings count mismatch"

## 4. Combine Text and Image Features

In [ ]:
# Combine text and image embeddings
print("Combining text and image features...")
train_combined_features = np.concatenate([train_text_embeddings, train_image_embeddings], axis=1)
test_combined_features = np.concatenate([test_text_embeddings, test_image_embeddings], axis=1)

print(f"Train combined features shape: {train_combined_features.shape}")
print(f"Test combined features shape: {test_combined_features.shape}")

# Extract target variable and apply log transform
train_prices = train_df['price'].values
train_log_prices = np.log1p(train_prices)  # log(1+x) transform

print(f"\nPrice statistics:")
print(f"Original - Min: {train_prices.min():.2f}, Max: {train_prices.max():.2f}, Mean: {train_prices.mean():.2f}")
print(f"Log-transformed - Min: {train_log_prices.min():.2f}, Max: {train_log_prices.max():.2f}, Mean: {train_log_prices.mean():.2f}")

## 5. Train-Validation Split (80-20)

In [ ]:
# 80-20 train-validation split
print("Splitting data into train and validation sets...")
X_train, X_val, y_train, y_val = train_test_split(
    train_combined_features, 
    train_log_prices,
    test_size=0.2,
    random_state=42
)

print(f"Training set size: {X_train.shape[0]}")
print(f"Validation set size: {X_val.shape[0]}")

# Standardize features
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_val_scaled = scaler.transform(X_val)
test_combined_features_scaled = scaler.transform(test_combined_features)

print("Feature scaling completed.")

## 6. Create PyTorch Dataset and DataLoader

In [ ]:
# Custom Dataset class
class PriceDataset(Dataset):
    def __init__(self, features, targets=None):
        self.features = torch.FloatTensor(features)
        self.targets = torch.FloatTensor(targets) if targets is not None else None
        
    def __len__(self):
        return len(self.features)
    
    def __getitem__(self, idx):
        if self.targets is not None:
            return self.features[idx], self.targets[idx]
        return self.features[idx]

# Create datasets
train_dataset = PriceDataset(X_train_scaled, y_train)
val_dataset = PriceDataset(X_val_scaled, y_val)
test_dataset = PriceDataset(test_combined_features_scaled)

# Create dataloaders
BATCH_SIZE = 64
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False)

print(f"Number of training batches: {len(train_loader)}")
print(f"Number of validation batches: {len(val_loader)}")
print(f"Number of test batches: {len(test_loader)}")

## 7. Define Informer Model Architecture

In [ ]:
# Informer Model Components

class PositionalEncoding(nn.Module):
    """Positional encoding for transformer"""
    def __init__(self, d_model, max_len=5000):
        super(PositionalEncoding, self).__init__()
        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len, dtype=torch.float).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        pe = pe.unsqueeze(0)
        self.register_buffer('pe', pe)

    def forward(self, x):
        return x + self.pe[:, :x.size(1), :]


class ProbSparseAttention(nn.Module):
    """ProbSparse Self-Attention mechanism (key feature of Informer)"""
    def __init__(self, d_model, n_heads, factor=5, dropout=0.1):
        super(ProbSparseAttention, self).__init__()
        self.d_model = d_model
        self.n_heads = n_heads
        self.d_k = d_model // n_heads
        self.factor = factor
        
        self.query_projection = nn.Linear(d_model, d_model)
        self.key_projection = nn.Linear(d_model, d_model)
        self.value_projection = nn.Linear(d_model, d_model)
        self.out_projection = nn.Linear(d_model, d_model)
        self.dropout = nn.Dropout(dropout)
        
    def forward(self, queries, keys, values, attn_mask=None):
        B, L, _ = queries.shape
        _, S, _ = keys.shape
        H = self.n_heads
        
        queries = self.query_projection(queries).view(B, L, H, self.d_k)
        keys = self.key_projection(keys).view(B, S, H, self.d_k)
        values = self.value_projection(values).view(B, S, H, self.d_k)
        
        # Transpose for attention computation
        queries = queries.transpose(1, 2)  # [B, H, L, d_k]
        keys = keys.transpose(1, 2)
        values = values.transpose(1, 2)
        
        # ProbSparse sampling (simplified version - using top-k queries)
        U_part = min(self.factor * int(math.ceil(math.log(L))), L)
        u = min(self.factor * int(math.ceil(math.log(S))), S)
        
        # Compute attention scores
        scores = torch.matmul(queries, keys.transpose(-2, -1)) / math.sqrt(self.d_k)
        
        if attn_mask is not None:
            scores = scores.masked_fill(attn_mask == 0, -1e9)
        
        attn = torch.softmax(scores, dim=-1)
        attn = self.dropout(attn)
        
        context = torch.matmul(attn, values)
        context = context.transpose(1, 2).contiguous().view(B, L, -1)
        
        return self.out_projection(context)


class EncoderLayer(nn.Module):
    """Informer Encoder Layer"""
    def __init__(self, d_model, n_heads, d_ff, factor=5, dropout=0.1):
        super(EncoderLayer, self).__init__()
        self.attention = ProbSparseAttention(d_model, n_heads, factor, dropout)
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)
        self.ffn = nn.Sequential(
            nn.Linear(d_model, d_ff),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(d_ff, d_model),
            nn.Dropout(dropout)
        )
        self.dropout = nn.Dropout(dropout)
        
    def forward(self, x, attn_mask=None):
        # Multi-head attention
        new_x = self.attention(x, x, x, attn_mask)
        x = x + self.dropout(new_x)
        x = self.norm1(x)
        
        # Feed-forward network
        new_x = self.ffn(x)
        x = x + new_x
        x = self.norm2(x)
        
        return x


class Encoder(nn.Module):
    """Informer Encoder"""
    def __init__(self, n_layers, d_model, n_heads, d_ff, factor=5, dropout=0.1):
        super(Encoder, self).__init__()
        self.layers = nn.ModuleList([
            EncoderLayer(d_model, n_heads, d_ff, factor, dropout)
            for _ in range(n_layers)
        ])
        self.norm = nn.LayerNorm(d_model)
        
    def forward(self, x, attn_mask=None):
        for layer in self.layers:
            x = layer(x, attn_mask)
        return self.norm(x)


class InformerPricePredictor(nn.Module):
    """Informer-based Price Predictor"""
    def __init__(self, input_dim, d_model=256, n_heads=8, n_layers=3, d_ff=512, 
                 factor=5, dropout=0.1, seq_len=1):
        super(InformerPricePredictor, self).__init__()
        
        self.input_dim = input_dim
        self.d_model = d_model
        self.seq_len = seq_len
        
        # Input embedding
        self.input_embedding = nn.Linear(input_dim, d_model)
        self.positional_encoding = PositionalEncoding(d_model, max_len=seq_len)
        
        # Informer encoder
        self.encoder = Encoder(n_layers, d_model, n_heads, d_ff, factor, dropout)
        
        # Attention pooling
        self.attention_pool = nn.Sequential(
            nn.Linear(d_model, 1),
            nn.Tanh()
        )
        
        # Output projection
        self.output_projection = nn.Sequential(
            nn.Linear(d_model, d_model // 2),
            nn.LayerNorm(d_model // 2),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(d_model // 2, d_model // 4),
            nn.LayerNorm(d_model // 4),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(d_model // 4, 1)
        )
        
    def forward(self, x):
        # Input embedding
        x = self.input_embedding(x)  # [B, d_model]
        x = x.unsqueeze(1)  # [B, 1, d_model] - add sequence dimension
        
        # Add positional encoding
        x = self.positional_encoding(x)
        
        # Encoder
        enc_out = self.encoder(x)  # [B, seq_len, d_model]
        
        # Attention pooling
        attn_weights = torch.softmax(self.attention_pool(enc_out), dim=1)
        pooled = torch.sum(attn_weights * enc_out, dim=1)  # [B, d_model]
        
        # Output projection
        output = self.output_projection(pooled)
        
        return output.squeeze(-1)


# Initialize model with default parameters (will be optimized by Optuna)
input_dim = X_train_scaled.shape[1]
model = InformerPricePredictor(
    input_dim=input_dim,
    d_model=256,
    n_heads=8,
    n_layers=3,
    d_ff=512,
    factor=5,
    dropout=0.1
)
model = model.to(device)

print(f"Informer Model initialized with input dimension: {input_dim}")
print(f"\nModel architecture:")
print(model)
print(f"\nTotal parameters: {sum(p.numel() for p in model.parameters()):,}")

## 8. Define Evaluation Metrics

In [ ]:
# Evaluation metrics functions
def calculate_smape(y_true, y_pred):
    """Calculate Symmetric Mean Absolute Percentage Error"""
    denominator = (np.abs(y_true) + np.abs(y_pred))
    diff = np.abs(y_true - y_pred) / denominator
    diff[denominator == 0] = 0  # Handle division by zero
    return 100 * np.mean(diff)

def calculate_mape(y_true, y_pred):
    """Calculate Mean Absolute Percentage Error"""
    mask = y_true != 0
    return 100 * np.mean(np.abs((y_true[mask] - y_pred[mask]) / y_true[mask]))

def evaluate_model(y_true, y_pred, dataset_name="Validation"):
    """Calculate and display all evaluation metrics"""
    mse = mean_squared_error(y_true, y_pred)
    mae = mean_absolute_error(y_true, y_pred)
    r2 = r2_score(y_true, y_pred)
    smape = calculate_smape(y_true, y_pred)
    mape = calculate_mape(y_true, y_pred)
    
    print(f"\n{dataset_name} Set Metrics:")
    print(f"{'='*50}")
    print(f"MSE:    {mse:.4f}")
    print(f"MAE:    {mae:.4f}")
    print(f"SMAPE:  {smape:.4f}%")
    print(f"MAPE:   {mape:.4f}%")
    print(f"R²:     {r2:.4f}")
    print(f"{'='*50}")
    
    return {
        'MSE': mse,
        'MAE': mae,
        'SMAPE': smape,
        'MAPE': mape,
        'R2': r2
    }

print("Evaluation metrics functions defined.")

## 9. Hyperparameter Tuning with Optuna

In [ ]:
# Define Optuna objective function for Informer
def objective(trial):
    """
    Optuna objective function to optimize Informer hyperparameters
    Returns validation SMAPE (lower is better)
    """
    
    # Suggest Informer-specific hyperparameters
    d_model = trial.suggest_categorical('d_model', [128, 256, 512])
    n_heads = trial.suggest_categorical('n_heads', [4, 8, 16])
    
    # Ensure d_model is divisible by n_heads
    while d_model % n_heads != 0:
        n_heads = trial.suggest_categorical('n_heads', [4, 8, 16])
    
    n_layers = trial.suggest_int('n_layers', 2, 6)
    d_ff = trial.suggest_int('d_ff', 256, 2048, step=256)
    factor = trial.suggest_int('factor', 3, 7)
    dropout = trial.suggest_float('dropout', 0.1, 0.5)
    learning_rate = trial.suggest_float('learning_rate', 1e-5, 1e-3, log=True)
    batch_size = trial.suggest_categorical('batch_size', [32, 64, 128])
    weight_decay = trial.suggest_float('weight_decay', 1e-6, 1e-4, log=True)
    
    # Create data loaders with suggested batch size
    train_dataset_trial = PriceDataset(X_train_scaled, y_train)
    val_dataset_trial = PriceDataset(X_val_scaled, y_val)
    
    train_loader_trial = DataLoader(train_dataset_trial, batch_size=batch_size, shuffle=True)
    val_loader_trial = DataLoader(val_dataset_trial, batch_size=batch_size, shuffle=False)
    
    # Initialize Informer model with suggested hyperparameters
    model_trial = InformerPricePredictor(
        input_dim=input_dim,
        d_model=d_model,
        n_heads=n_heads,
        n_layers=n_layers,
        d_ff=d_ff,
        factor=factor,
        dropout=dropout
    ).to(device)
    
    # Define optimizer and loss
    criterion_trial = nn.MSELoss()
    optimizer_trial = optim.AdamW(
        model_trial.parameters(),
        lr=learning_rate,
        weight_decay=weight_decay
    )
    
    # Training loop (fewer epochs for faster tuning)
    num_epochs_trial = 15
    best_val_smape = float('inf')
    patience = 5
    patience_counter = 0
    
    for epoch in range(num_epochs_trial):
        # Training phase
        model_trial.train()
        train_loss = 0.0
        
        for features, targets in train_loader_trial:
            features = features.to(device)
            targets = targets.to(device)
            
            optimizer_trial.zero_grad()
            outputs = model_trial(features)
            loss = criterion_trial(outputs, targets)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model_trial.parameters(), max_norm=1.0)
            optimizer_trial.step()
            
            train_loss += loss.item()
        
        # Validation phase
        model_trial.eval()
        val_predictions_log = []
        val_targets_log = []
        
        with torch.no_grad():
            for features, targets in val_loader_trial:
                features = features.to(device)
                outputs = model_trial(features)
                val_predictions_log.extend(outputs.cpu().numpy())
                val_targets_log.extend(targets.cpu().numpy())
        
        # Convert to numpy arrays
        val_predictions_log = np.array(val_predictions_log)
        val_targets_log = np.array(val_targets_log)
        
        # Inverse transform to original scale
        val_predictions = np.expm1(val_predictions_log)
        val_targets = np.expm1(val_targets_log)
        
        # Calculate SMAPE
        current_smape = calculate_smape(val_targets, val_predictions)
        
        # Report intermediate value for pruning
        trial.report(current_smape, epoch)
        
        # Handle pruning based on the intermediate value
        if trial.should_prune():
            raise optuna.TrialPruned()
        
        # Early stopping
        if current_smape < best_val_smape:
            best_val_smape = current_smape
            patience_counter = 0
        else:
            patience_counter += 1
            if patience_counter >= patience:
                break
    
    return best_val_smape

print("Optuna objective function for Informer defined.")

In [ ]:
# Create and run Optuna study
print("Starting Optuna hyperparameter optimization...")
print("This may take a while depending on n_trials...\n")

# Create study (minimize SMAPE)
study = optuna.create_study(
    direction='minimize',
    pruner=optuna.pruners.MedianPruner(n_startup_trials=5, n_warmup_steps=5),
    sampler=optuna.samplers.TPESampler(seed=42)
)

# Optimize
N_TRIALS = 30  # Adjust this based on your compute resources and time
study.optimize(objective, n_trials=N_TRIALS, show_progress_bar=True)

print("\n" + "="*70)
print("OPTUNA OPTIMIZATION COMPLETED!")
print("="*70)

# Best trial
best_trial = study.best_trial
print(f"\n🏆 Best Trial: {best_trial.number}")
print(f"📊 Best SMAPE: {best_trial.value:.4f}%")
print(f"\n🔧 Best Hyperparameters:")
for key, value in best_trial.params.items():
    print(f"  - {key}: {value}")

# Save best hyperparameters
best_params = best_trial.params
print(f"\n✅ Best hyperparameters saved!")

In [ ]:
# Visualize optimization results
import matplotlib.pyplot as plt

# Plot optimization history
fig1 = optuna.visualization.matplotlib.plot_optimization_history(study)
plt.title("Optimization History (SMAPE over Trials)")
plt.tight_layout()
plt.show()

# Plot parameter importances
fig2 = optuna.visualization.matplotlib.plot_param_importances(study)
plt.title("Hyperparameter Importances")
plt.tight_layout()
plt.show()

# Plot parallel coordinate
try:
    fig3 = optuna.visualization.matplotlib.plot_parallel_coordinate(study)
    plt.title("Parallel Coordinate Plot")
    plt.tight_layout()
    plt.show()
except:
    print("Parallel coordinate plot not available")

# Show trial statistics
print("\n📈 Trial Statistics:")
print(f"Number of finished trials: {len(study.trials)}")
print(f"Number of pruned trials: {len([t for t in study.trials if t.state == optuna.trial.TrialState.PRUNED])}")
print(f"Number of complete trials: {len([t for t in study.trials if t.state == optuna.trial.TrialState.COMPLETE])}")

## 10. Train Final Model with Optimized Hyperparameters

In [ ]:
# Training configuration with optimized hyperparameters
print("Initializing final Informer model with best hyperparameters from Optuna...")

# Create data loaders with optimized batch size
BATCH_SIZE = best_params['batch_size']
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False)

# Initialize Informer model with best hyperparameters
model = InformerPricePredictor(
    input_dim=input_dim,
    d_model=best_params['d_model'],
    n_heads=best_params['n_heads'],
    n_layers=best_params['n_layers'],
    d_ff=best_params['d_ff'],
    factor=best_params['factor'],
    dropout=best_params['dropout']
).to(device)

print(f"\nInformer Model initialized with optimized hyperparameters:")
print(f"  - d_model: {best_params['d_model']}")
print(f"  - n_heads: {best_params['n_heads']}")
print(f"  - n_layers: {best_params['n_layers']}")
print(f"  - d_ff (feed-forward dim): {best_params['d_ff']}")
print(f"  - factor (ProbSparse): {best_params['factor']}")
print(f"  - dropout: {best_params['dropout']}")
print(f"  - batch_size: {best_params['batch_size']}")
print(f"  - learning_rate: {best_params['learning_rate']}")
print(f"  - weight_decay: {best_params['weight_decay']}")
print(f"\nTotal parameters: {sum(p.numel() for p in model.parameters()):,}")

criterion = nn.MSELoss()
optimizer = optim.AdamW(
    model.parameters(),
    lr=best_params['learning_rate'],
    weight_decay=best_params['weight_decay']
)
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=3, verbose=True)

NUM_EPOCHS = 100  # More epochs for final training
best_val_loss = float('inf')
patience = 15
patience_counter = 0

# Training history
train_losses = []
val_losses = []

print(f"\nStarting final training...")
print(f"Device: {device}")
print(f"Epochs: {NUM_EPOCHS}")
print(f"Batch size: {BATCH_SIZE}\n")

for epoch in range(NUM_EPOCHS):
    # Training phase
    model.train()
    train_loss = 0.0
    
    for features, targets in train_loader:
        features = features.to(device)
        targets = targets.to(device)
        
        # Forward pass
        optimizer.zero_grad()
        outputs = model(features)
        loss = criterion(outputs, targets)
        
        # Backward pass
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        
        train_loss += loss.item()
    
    avg_train_loss = train_loss / len(train_loader)
    train_losses.append(avg_train_loss)
    
    # Validation phase
    model.eval()
    val_loss = 0.0
    
    with torch.no_grad():
        for features, targets in val_loader:
            features = features.to(device)
            targets = targets.to(device)
            
            outputs = model(features)
            loss = criterion(outputs, targets)
            val_loss += loss.item()
    
    avg_val_loss = val_loss / len(val_loader)
    val_losses.append(avg_val_loss)
    
    # Learning rate scheduling
    scheduler.step(avg_val_loss)
    
    # Print progress
    if (epoch + 1) % 5 == 0 or epoch == 0:
        print(f"Epoch [{epoch+1}/{NUM_EPOCHS}] - Train Loss: {avg_train_loss:.4f}, Val Loss: {avg_val_loss:.4f}")
    
    # Early stopping
    if avg_val_loss < best_val_loss:
        best_val_loss = avg_val_loss
        patience_counter = 0
        # Save best model
        torch.save(model.state_dict(), 'best_informer_model_optimized.pth')
    else:
        patience_counter += 1
        if patience_counter >= patience:
            print(f"\nEarly stopping triggered at epoch {epoch+1}")
            break

print("\nTraining completed!")
print(f"Best validation loss: {best_val_loss:.4f}")

In [ ]:
# Plot training history
import matplotlib.pyplot as plt

plt.figure(figsize=(10, 6))
plt.plot(train_losses, label='Train Loss', linewidth=2)
plt.plot(val_losses, label='Validation Loss', linewidth=2)
plt.xlabel('Epoch', fontsize=12)
plt.ylabel('Loss (MSE)', fontsize=12)
plt.title('Training and Validation Loss Over Epochs', fontsize=14)
plt.legend(fontsize=11)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print(f"Final train loss: {train_losses[-1]:.4f}")
print(f"Final validation loss: {val_losses[-1]:.4f}")

## 11. Validation Set Evaluation

In [ ]:
# Load best Informer model
model.load_state_dict(torch.load('best_informer_model_optimized.pth'))
model.eval()

# Make predictions on validation set
val_predictions_log = []

with torch.no_grad():
    for features, _ in val_loader:
        features = features.to(device)
        outputs = model(features)
        val_predictions_log.extend(outputs.cpu().numpy())

val_predictions_log = np.array(val_predictions_log)

# Inverse transform: convert from log space to original price space
val_predictions = np.expm1(val_predictions_log)  # exp(x) - 1
y_val_original = np.expm1(y_val)  # Convert true values back too

# Evaluate on validation set
val_metrics = evaluate_model(y_val_original, val_predictions, dataset_name="Validation")

# Show some prediction examples
print("\n\nSample Predictions vs Actual Prices:")
print(f"{'Actual Price':<15} {'Predicted Price':<15} {'Difference':<15} {'% Error':<10}")
print("-" * 60)
for i in range(min(10, len(y_val_original))):
    actual = y_val_original[i]
    predicted = val_predictions[i]
    diff = predicted - actual
    pct_error = 100 * abs(diff) / actual if actual != 0 else 0
    print(f"{actual:<15.2f} {predicted:<15.2f} {diff:<15.2f} {pct_error:<10.2f}%")

In [ ]:
# Visualize predictions vs actual
plt.figure(figsize=(12, 5))

# Scatter plot
plt.subplot(1, 2, 1)
plt.scatter(y_val_original, val_predictions, alpha=0.5, s=10)
plt.plot([y_val_original.min(), y_val_original.max()], 
         [y_val_original.min(), y_val_original.max()], 
         'r--', linewidth=2, label='Perfect Prediction')
plt.xlabel('Actual Price', fontsize=12)
plt.ylabel('Predicted Price', fontsize=12)
plt.title('Validation Set: Predicted vs Actual Prices', fontsize=14)
plt.legend(fontsize=10)
plt.grid(True, alpha=0.3)

# Residual plot
plt.subplot(1, 2, 2)
residuals = val_predictions - y_val_original
plt.scatter(y_val_original, residuals, alpha=0.5, s=10)
plt.axhline(y=0, color='r', linestyle='--', linewidth=2)
plt.xlabel('Actual Price', fontsize=12)
plt.ylabel('Residuals (Predicted - Actual)', fontsize=12)
plt.title('Residual Plot', fontsize=14)
plt.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 12. Generate Test Set Predictions

In [ ]:
# Generate predictions on test set
print("Generating predictions on test set...")
test_predictions_log = []

model.eval()
with torch.no_grad():
    for features in test_loader:
        features = features.to(device)
        outputs = model(features)
        test_predictions_log.extend(outputs.cpu().numpy())

test_predictions_log = np.array(test_predictions_log)

# Inverse transform: convert from log space to original price space
test_predictions = np.expm1(test_predictions_log)

print(f"Number of test predictions: {len(test_predictions)}")
print(f"Prediction statistics:")
print(f"  Min:  {test_predictions.min():.2f}")
print(f"  Max:  {test_predictions.max():.2f}")
print(f"  Mean: {test_predictions.mean():.2f}")
print(f"  Median: {np.median(test_predictions):.2f}")

## 13. Create Submission File

In [ ]:
# Create submission dataframe
submission_df = pd.DataFrame({
    'sample_id': test_df['sample_id'],
    'price': test_predictions
})

# Save to CSV
submission_filename = 'test_predictions.csv'
submission_df.to_csv(submission_filename, index=False)

print(f"Submission file saved as: {submission_filename}")
print(f"\nSubmission file preview:")
print(submission_df.head(10))
print(f"\nTotal predictions: {len(submission_df)}")

## 14. Summary and Results

In [ ]:
# Print comprehensive summary
print("=" * 70)
print("APPROACH 9: INFORMER WITH TEXT + IMAGE EMBEDDINGS - FINAL SUMMARY")
print("=" * 70)

print("\n🔬 HYPERPARAMETER OPTIMIZATION:")
print(f"  - Optimization method: Optuna (TPE Sampler)")
print(f"  - Number of trials: {len(study.trials)}")
print(f"  - Best trial: #{best_trial.number}")
print(f"  - Optimized metric: SMAPE")
print(f"  - Best SMAPE from tuning: {best_trial.value:.4f}%")

print("\n🔧 OPTIMIZED HYPERPARAMETERS:")
for key, value in best_params.items():
    print(f"  - {key}: {value}")

print("\n📊 MODEL ARCHITECTURE (INFORMER):")
print("  - Input: Combined Text (TF-IDF) + Image (EfficientNet-B0) embeddings")
print("  - Text embedding dimension: 512")
print(f"  - Image embedding dimension: {train_image_embeddings.shape[1]}")
print(f"  - Total input dimension: {input_dim}")
print(f"  - Transformer d_model: {best_params['d_model']}")
print(f"  - Number of attention heads: {best_params['n_heads']}")
print(f"  - Number of encoder layers: {best_params['n_layers']}")
print(f"  - Feed-forward dimension: {best_params['d_ff']}")
print(f"  - ProbSparse factor: {best_params['factor']}")
print("  - Self-attention: ProbSparse (efficient)")
print("  - Positional encoding: Yes")
print("  - Attention pooling: Yes")
print(f"  - Dropout: {best_params['dropout']}")
print(f"  - Total parameters: {sum(p.numel() for p in model.parameters()):,}")

print("\n📈 DATASET SPLIT:")
print(f"  - Total training samples: {len(train_df)}")
print(f"  - Training set: {X_train.shape[0]} samples (80%)")
print(f"  - Validation set: {X_val.shape[0]} samples (20%)")
print(f"  - Test set: {len(test_df)} samples")

print("\n🎯 VALIDATION SET PERFORMANCE:")
print(f"  - MSE:    {val_metrics['MSE']:.4f}")
print(f"  - MAE:    {val_metrics['MAE']:.4f}")
print(f"  - SMAPE:  {val_metrics['SMAPE']:.4f}% ⭐ (Primary Metric)")
print(f"  - MAPE:   {val_metrics['MAPE']:.4f}%")
print(f"  - R²:     {val_metrics['R2']:.4f}")

print("\n💾 OUTPUT FILES:")
print(f"  - Model weights: best_informer_model_optimized.pth")
print(f"  - Test predictions: test_predictions.csv")

print("\n✅ KEY FEATURES:")
print("  - Informer architecture (state-of-the-art for time series)")
print("  - ProbSparse self-attention (O(L log L) complexity)")
print("  - Optuna hyperparameter optimization (30 trials)")
print("  - Multi-head attention mechanism")
print("  - Positional encoding for sequence modeling")
print("  - Log transformation applied to prices")
print("  - Feature standardization (StandardScaler)")
print("  - Early stopping with patience=15")
print("  - Learning rate scheduling (ReduceLROnPlateau)")
print("  - Gradient clipping (max_norm=1.0)")
print("  - Layer normalization")
print("  - GELU activation functions")
print("  - Median pruner for early trial termination")

print("\n🚀 INFORMER ADVANTAGES:")
print("  - Efficient self-attention (ProbSparse)")
print("  - Better long-range dependency modeling")
print("  - Reduced memory footprint vs standard Transformer")
print("  - State-of-the-art for forecasting tasks")
print("  - Handles multivariate features effectively")

print("\n📚 INFORMER DETAILS:")
print("  - Paper: 'Informer: Beyond Efficient Transformer for Long Sequence Time-Series Forecasting'")
print("  - Key innovation: ProbSparse attention reduces complexity from O(L²) to O(L log L)")
print("  - Attention distilling: Progressive reduction of sequence length")

print("\n" + "=" * 70)
print("Notebook execution complete! ✨")
print("=" * 70)